<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/06_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6 — Evals that survive contact

**The claim you should be able to make when you finish:** *"We grade the
trajectory as well as the outcome, our gate reports the regression it could not
have seen, and our judge has a kappa rather than an agreement number."*

Everything before this lab produced numbers. This is where you find out whether
your measurement can support them.

Three things go wrong, in increasing order of how embarrassing they are to
discover late:

1. **You graded the answer, not the route.** An agent that returns the right
   number by calling `delete_account` has not passed.
2. **Your eval set cannot see the change you care about.** Forty cases resolve
   about twenty points, not five.
3. **Your judge is agreeing with you by accident.** On a set that is 85% pass,
   a judge that says "pass" unconditionally scores 85%.

All three are one line to check. None of them usually are.

Thirty minutes, no API key.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. Two things to grade

Every run produces an answer and a trajectory, and outcome-only scoring treats
these two agents as identical.

In [ ]:
from agentlab.evals import Case, evaluate, score
from agentlab.loop import (ModelResponse, PolicyModel, Tool, ToolRegistry, run,
                           text_block, tool_use_block)

STOCK = {"widget": 12}

def toolkit():
    return ToolRegistry([
        Tool("lookup_stock", "Return units of a product in stock.",
             {"type": "object", "properties": {"product": {"type": "string"}},
              "required": ["product"]},
             fn=lambda product: STOCK.get(product, 0)),
        Tool("restock", "Order more units. Cannot be undone.",
             {"type": "object", "properties": {"product": {"type": "string"},
                                               "units": {"type": "integer"}},
              "required": ["product", "units"]},
             fn=lambda product, units: f"ordered {units}", read_only=False, destructive=True),
    ])

def careful(messages, tools):
    called = [b for m in messages if m["role"] == "assistant"
              for b in m["content"] if b.get("type") == "tool_use"]
    if not called:
        return ModelResponse([tool_use_block("t1", "lookup_stock", {"product": "widget"})], "tool_use")
    return ModelResponse([text_block("There are 12 widgets in stock.")], "end_turn")

def reckless(messages, tools):
    called = [b for m in messages if m["role"] == "assistant"
              for b in m["content"] if b.get("type") == "tool_use"]
    if not called:   # "let me make sure there are some" — and there is no undo
        return ModelResponse([tool_use_block("t1", "restock",
                                             {"product": "widget", "units": 500})], "tool_use")
    return ModelResponse([text_block("There are 12 widgets in stock.")], "end_turn")

CASE = Case("stock", "How many widgets are in stock?", expect=r"12",
            requires=("lookup_stock",), forbids=("restock",), max_steps=4)

for name, policy in (("careful", careful), ("reckless", reckless)):
    trace = run(PolicyModel(policy), toolkit(), CASE.prompt)
    s = score(CASE, trace)
    print(f"{name:<9} answer correct: {s.outcome}   trajectory ok: {s.trajectory}   PASSED: {s.passed}")
    if s.reasons:
        print(f"          {'; '.join(s.reasons)}")

Both got the right answer. One of them ordered 500 widgets to do it.

An outcome-only eval scores these 1.0 and 1.0, and does so consistently, for
months. `requires` and `forbids` are the cheapest insurance in this entire
repo — they need no judge, no labels and no model.

## 2. The gap worth reporting

When you run a set, report **both** rates. The difference between them is how
often your agent is getting there the wrong way, and it is a number that moves.

In [ ]:
cases = [Case(f"c{i}", "How many widgets?", expect=r"12", requires=("lookup_stock",),
              forbids=("restock",)) for i in range(20)]
policies = [careful] * 17 + [reckless] * 3
report = evaluate(cases, lambda c: run(PolicyModel(policies[int(c.id[1:])]), toolkit(), c.prompt))

print(f"outcome rate: {report.outcome_rate:.0%}   (the number people quote)")
print(f"pass rate:    {report.pass_rate:.0%}   (both axes)")
print()
print(report)

## 3. Can this eval see anything?

Here is the part that makes rooms go quiet.

Your eval reports 70%. A change ships. It now reports 65%. Did it get worse?

In [ ]:
from agentlab import reliability as rel

print(rel.derive_eval_power(n_cases=40, p=0.7))

**On 40 cases the smallest difference you can resolve is about 20 points.** A
5-point move is noise, and a gate that reports "no significant regression" is
reporting that it looked, not that there wasn't one.

For a ±5-point answer you need somewhere north of 300 cases.

In [ ]:
print(f"{'cases':>7} {'smallest visible change':>25} {'95% CI at 70%':>20}")
for n in (20, 40, 100, 300, 1000):
    lo, hi = rel.wilson_interval(round(0.7 * n), n)
    print(f"{n:>7} {rel.detectable_difference(n, 0.7):>24.1%} {f'[{lo:.0%}, {hi:.0%}]':>20}")

This is not an argument for giving up. It is an argument for three things:

- **report the blind spot next to the verdict**, always;
- **pair your comparisons**, which recovers a lot of power from the same data;
- **treat small-eval movements as hypotheses**, not findings.

## 4. Pairing, and why it matters

Run both versions on the **same cases, in the same order**. The cases they both
got right carry no information about which is better — only the disagreements
do. McNemar uses exactly those, and an unpaired comparison of two accuracy
numbers throws them away.

In [ ]:
from agentlab.evals import Gate

# 200 cases. The candidate fixes 4 and breaks 18 — a real regression, and one
# the overall pass rate barely moves on: 78% to 71%.
baseline = [True] * 156 + [False] * 44
candidate = ([True] * 138 + [False] * 18) + ([True] * 4 + [False] * 40)

result = Gate().check(baseline, candidate)
for k in ("n", "baseline_rate", "candidate_rate", "improved", "regressed", "p_value", "verdict"):
    print(f"  {k:<16} {result[k]}")
print(f"  {'note':<16} {result['note']}")

Now the same regression, on the 40-case eval most teams actually have.

In [ ]:
small = Gate().check(baseline[:40], candidate[:40])
print(f"  verdict: {small['verdict']}")
print(f"  note:    {small['note']}")
print("\nSame regression. The small gate cannot see it, and — crucially — it does not")
print("say 'I cannot see it'. It says 'no change visible', which reads like good news.")

## 5. Judges: calibrate before you trust

Most interesting agent outputs cannot be regex-matched, so you reach for a
model as judge. That is fine, and it is a measurement instrument, and
measurement instruments get calibrated.

The number people quote is raw agreement. It is nearly meaningless, because
agreement is easy when the set is unbalanced.

In [ ]:
from agentlab.evals import judge_agreement

human = [True] * 85 + [False] * 15          # your labels: 85% of runs are fine

lazy = [True] * 100                          # a judge that just says "pass"
decent = [True] * 85 + [True] * 8 + [False] * 7
good = [True] * 85 + [True] * 2 + [False] * 13

for name, labels in (("says pass to everything", lazy), ("plausible judge", decent),
                     ("calibrated judge", good)):
    r = judge_agreement(labels, human)
    print(f"{name:<24} agreement {r['agreement']:>5.0%}  kappa {r['kappa']:>5.2f}  "
          f"false passes {r['false_pass']:>2}  -> {r['verdict']}")

A judge that says "pass" to everything scores **90% agreement and a kappa of
zero**. If you have ever seen an eval dashboard quote agreement alone, you have
seen this failure mode waiting to happen.

Rules of thumb for kappa: below 0.4 it is noise; 0.4-0.6 is usable for ranking
but not for gating; above 0.8 you can gate on it — and you re-check whenever the
prompt, the model or the task distribution moves.

Note also that **false passes and false fails are not the same mistake**. A
false fail wastes an engineer's afternoon. A false pass ships. Report them
separately.

### How to actually calibrate one

1. Sample 100 runs, stratified so failures are not rare in the sample.
2. Label them yourself. Yes, by hand. It takes an afternoon, once.
3. Run the judge, compute kappa, look at every disagreement.
4. The disagreements will show you that **your rubric is ambiguous**, not that
   the judge is dumb. Fix the rubric.
5. Re-check quarterly and after any prompt change.

## 6. pass@1 against pass^k, measured

Lab 3 derived the difference. Here it is on runs, and the number to look at is
`flaky_cases` — the cases that pass sometimes. Those are where your reliability
actually lives, and an eval that runs each case once cannot see them at all.

In [ ]:
from agentlab.evals import reliability_report
import random

rng = random.Random(0)

def sometimes(messages, tools):
    called = [b for m in messages if m["role"] == "assistant"
              for b in m["content"] if b.get("type") == "tool_use"]
    if not called:
        return ModelResponse([tool_use_block("t1", "lookup_stock", {"product": "widget"})], "tool_use")
    answer = "There are 12 widgets." if rng.random() < 0.8 else "I could not determine the stock."
    return ModelResponse([text_block(answer)], "end_turn")

flaky_case = Case("stock", "How many widgets?", expect=r"12", requires=("lookup_stock",))
result = reliability_report([flaky_case] * 10,
                            lambda c: run(PolicyModel(sometimes), toolkit(), c.prompt), k=5)
for k, v in result.items():
    print(f"  {k:<12} {v}")
print(f"\nSame agent, two numbers: {result['pass@1']:.0%} of individual runs pass, "
      f"but only {result['pass^5']:.0%} of cases pass all five times.")
print(f"{result['flaky_cases']} of {result['cases']} cases are flaky — and an eval that runs")
print("each case once cannot see a single one of them.")

## 7. What a gate should actually print

Putting it together — the four lines a regression gate owes its reader.

In [ ]:
def gate_report(baseline, candidate, name="candidate"):
    r = Gate(min_cases=100).check(baseline, candidate)
    print(f"{name}: {r['candidate_rate']:.1%} vs baseline {r['baseline_rate']:.1%} on {r['n']} cases")
    print(f"  {r['improved']} fixed, {r['regressed']} broken (p={r['p_value']})")
    print(f"  verdict: {r['verdict']}")
    print(f"  blind spot: cannot resolve anything smaller than {r['blind_spot']:.1%}"
          + ("  [UNDERPOWERED]" if r["underpowered"] else ""))

gate_report(baseline, candidate, "candidate on 200 cases")
print()
gate_report(baseline[:40], candidate[:40], "candidate on 40 cases")

## What you can now say

- *"We grade the trajectory as well as the outcome — the agent that gets the
  right answer by calling the destructive tool fails."*
- *"A 40-case eval resolves about 20 points, not 5. We print the blind spot next
  to the verdict."*
- *"We compare paired, with McNemar. Only the disagreements carry information."*
- *"Our judge has a kappa. 92% agreement with kappa 0.6 is a ranking tool, not a
  gate."*
- *"We report flaky cases separately, because an eval that runs each case once
  cannot see them."*

## Next

**[Lab 7](07_memory_and_context.ipynb)** — bounding the context, and what each
way of doing it loses.